In [14]:
import os
import librosa
import numpy as np
import random

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.regularizers import l2

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [15]:
asthma_data = "./Datasets/Asthma"
copd_data = "./Datasets/COPD4"
healthy_data = "./Datasets/Healthy"

print("Asthma files:", len(os.listdir(asthma_data)))
print("COPD files:", len(os.listdir(copd_data)))
print("Healthy files:", len(os.listdir(healthy_data)))

Asthma files: 96
COPD files: 112
Healthy files: 112


In [16]:
def augment_noise(audio, sr=None):
    noise = np.random.randn(len(audio))
    return audio + 0.005 * noise

def augment_pitch(audio, sr):
    return librosa.effects.pitch_shift(audio, sr=sr, n_steps=random.uniform(-2, 2))

def augment_speed(audio, sr):
    speed = random.uniform(0.9, 1.1)
    return librosa.effects.time_stretch(audio, rate=speed)

def apply_augment(audio, sr=22050):
    funcs = [
        lambda y: augment_noise(y),
        lambda y: augment_pitch(y, sr),
        lambda y: augment_speed(y, sr)
    ]
    func = random.choice(funcs)
    return func(audio)

In [17]:
def extract_mfcc(audio, sr=22050):
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=40,
        n_fft=2048,
        hop_length=512
    )

    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std  = np.std(mfcc, axis=1)

    return np.concatenate([mfcc_mean, mfcc_std])


In [18]:
def split_audio(audio, sr, win_sec=3, overlap_ratio=0.2):
    win_len = int(win_sec * sr)
    hop_len = int(win_len * (1-overlap_ratio))
    segments = []

    for start in range(0, len(audio)-win_len, hop_len):
        segments.append(audio[start:start+win_len])

    return segments


In [19]:
class_folders = {
    "Asthma": asthma_data,
    "COPD": copd_data,
    "Healthy": healthy_data
}

all_files = []
for label, folder in class_folders.items():
    for f in os.listdir(folder):
        if f.endswith('.wav'):
            all_files.append((os.path.join(folder, f), label))

train_files, test_files = train_test_split(
    all_files, test_size=0.2, stratify=[x[1] for x in all_files], random_state=42
)

In [20]:
def prepare_data(file_list, augment=False):
    X, y = [], []

    for path, label in file_list:
        try:
            audio, sr = librosa.load(path, sr=22050)
            segments = split_audio(audio, sr)

            for seg in segments:
                feat = extract_mfcc(seg, sr)
                X.append(feat)
                y.append(label)

                if augment:
                    aug = apply_augment(seg)
                    X.append(extract_mfcc(aug, sr))
                    y.append(label)

        except Exception as e:
            print(f"Hata: {path} -> {e}")

    return np.array(X), np.array(y)



In [21]:
print("Veriler işleniyor (bu biraz sürebilir)...")
X_train_raw, y_train_raw = prepare_data(train_files)
X_test_raw, y_test_raw = prepare_data(test_files)

Veriler işleniyor (bu biraz sürebilir)...


In [22]:
print("Veriler hazırlanıyor...")

X_train, y_train = prepare_data(train_files, augment=True)
X_test,  y_test  = prepare_data(test_files, augment=False)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)


le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)


Veriler hazırlanıyor...
Train shape: (4126, 80)
Test shape : (514, 80)


In [23]:
def build_mlp(input_dim, layers):
    model = Sequential()

    for i, u in enumerate(layers):
        if i == 0:
            model.add(Dense(u, input_dim=input_dim, kernel_regularizer=l2(1e-4)))
        else:
            model.add(Dense(u, kernel_regularizer=l2(1e-4)))

        model.add(BatchNormalization())
        model.add(Activation("relu"))
        model.add(Dropout(0.3))

    model.add(Dense(3, activation="softmax"))

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [24]:


configs = {

    "1_layer": [[32],[64],[128],[256],[512]],

    "2_layer": [[32,64],[64,128],[128,256],[256,512]],

    "3_layer": [[32,64,128],[64,128,256],[128,256,512]]

}

In [25]:
results = []

for group, cfgs in configs.items():
    print(f"\n--- {group} Test Ediliyor ---")
    for units in cfgs:
        model = build_mlp(X_train.shape[1], units)
        history = model.fit(
            X_train, y_train_enc,
            validation_data=(X_test, y_test_enc),
            epochs=100,
            batch_size=32,
            verbose=0
        )
        
        loss, acc = model.evaluate(X_test, y_test_enc, verbose=0)
        print(f"Layers {units} -> Test Acc: {acc:.4f}")
        results.append({"cfg": units, "acc": acc})


--- 1_layer Test Ediliyor ---


c:\Users\MONSTER\Desktop\LungSoundAnlysis\LungSoundAnlysis\venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Layers [32] -> Test Acc: 0.6362
Layers [64] -> Test Acc: 0.6595
Layers [128] -> Test Acc: 0.6712
Layers [256] -> Test Acc: 0.6790
Layers [512] -> Test Acc: 0.6751

--- 2_layer Test Ediliyor ---
Layers [32, 64] -> Test Acc: 0.6576
Layers [64, 128] -> Test Acc: 0.6712
Layers [128, 256] -> Test Acc: 0.6615
Layers [256, 512] -> Test Acc: 0.6498

--- 3_layer Test Ediliyor ---
Layers [32, 64, 128] -> Test Acc: 0.6440
Layers [64, 128, 256] -> Test Acc: 0.6498
Layers [128, 256, 512] -> Test Acc: 0.6634


In [26]:
best = max(results, key=lambda x: x["acc"])
print("\n🔥 EN İYİ MODEL 🔥")
print(best)



🔥 EN İYİ MODEL 🔥
{'cfg': [256], 'acc': 0.6789883375167847}
